# Hotels Reviews Dashboard
**Google Maps Reviews — Interactive Analysis**  
Data sourced via Apify Google Maps Reviews Scraper

In [ ]:
# ── Dependencies ──────────────────────────────────────────────
import os, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import ipywidgets as widgets
from wordcloud import WordCloud, STOPWORDS
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from collections import defaultdict
import spacy

warnings.filterwarnings('ignore')
pio.renderers.default = "notebook"



## 1. Load & Clean Data

In [26]:


def parse_experience_details(val):
    """Flatten the experience_details JSON array into a dict."""
    if pd.isna(val) or val == '':
        return {}
    try:
        items = json.loads(val)
        result = {}
        for item in items:
            name = item.get('name')
            value = item.get('value')
            if name and value is not None:
                result[name] = value
        return result
    except:
        return {}

def load_hotel_reviews(data_dir='data'):
    csv_files = glob.glob(os.path.join(data_dir, '*.csv'))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in '{data_dir}/'")

    dfs = []
    for path in csv_files:
        try:
            raw = pd.read_csv(path)
            dfs.append(raw)
            print(f"  ✅ {os.path.basename(path):60s} — {len(raw):>4} rows")
        except Exception as e:
            print(f"  ❌ {os.path.basename(path)} — skipped: {e}")

    combined = pd.concat(dfs, ignore_index=True)

    # ── Rename flat columns ───────────────────────────────────
    df = combined.rename(columns={
        'place_name':                         'hotel',
        'name':                               'reviewer',
        'rating':                             'stars',
        'review_text':                        'text',
        'review_translated_text':             'text_translated',
        'published_at_date':                  'date',
        'total_number_of_reviews_by_reviewer':'reviewer_reviews',
        'total_number_of_photos_by_reviewer': 'reviewer_photos',
        'review_origin':                      'source',
        'response_from_owner_text':           'owner_response',
    }).copy()

    # ── Parse dates ───────────────────────────────────────────
    df['date'] = pd.to_datetime(df['date'], utc=True).dt.tz_localize(None)
    df['year_month']  = df['date'].dt.to_period('M').dt.to_timestamp()
    df['year']        = df['date'].dt.year
    df['month']       = df['date'].dt.month

    # ── Parse nested experience_details ──────────────────────
    details = df['experience_details'].apply(parse_experience_details)
    details_df = pd.DataFrame(details.tolist())

    # Numeric ratings
    for col in ['Rooms', 'Service', 'Location']:
        df[f'rating_{col.lower()}'] = pd.to_numeric(
            details_df.get(col), errors='coerce'
        )

    # Categorical context
    df['trip_type']     = details_df.get('Trip type')
    df['travel_group']  = details_df.get('Travel group')
    df['highlights']    = details_df.get('Hotel highlights')

    # ── Keep useful columns ───────────────────────────────────
    df = df[[
        'place_id', 'hotel', 'reviewer', 'reviewer_id', 'stars', 'date',
        'year_month', 'year', 'month', 'text', 'text_translated',
        'is_local_guide', 'reviewer_reviews', 'reviewer_photos',
        'rating_rooms', 'rating_service', 'rating_location',
        'trip_type', 'travel_group', 'highlights',
        'owner_response', 'source'
    ]]

    df['stars'] = pd.to_numeric(df['stars'], errors='coerce')
    df.dropna(subset=['stars', 'date'], inplace=True)
    df['stars'] = df['stars'].astype(int)

    # ── Use translated text where original is missing ─────────
    df['text'] = df['text'].fillna(df['text_translated'])

    # ── Dedup ─────────────────────────────────────────────────
    before = len(df)
    df.drop_duplicates(subset=['hotel', 'reviewer', 'date'], inplace=True)
    dupes = before - len(df)

    print(f"\n📊 {len(df):,} reviews | {df['hotel'].nunique()} hotels")
    if dupes:
        print(f"🧹 Dropped {dupes} duplicates")
    print(f"📅 {df['date'].min().date()} → {df['date'].max().date()}")
    print(f"\nReviews per hotel:")
    print(df['hotel'].value_counts().to_string())
    print(f"\nDetailed ratings coverage:")
    for col in ['rating_rooms','rating_service','rating_location']:
        n = df[col].notna().sum()
        print(f"  {col}: {n:,} / {len(df):,} ({n/len(df)*100:.0f}%)")
    print(f"\nTrip type breakdown:")
    print(df['trip_type'].value_counts().to_string())
    print(f"\nTravel group breakdown:")
    print(df['travel_group'].value_counts().to_string())

    return df



In [27]:
# ── Load ──────────────────────────────────────────────────────
# ── Load ──────────────────────────────────────────────────────
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')
df = load_hotel_reviews(DATA_DIR)
df.head(10)

  ✅ banff_hotels_filtered.csv                                    — 6756 rows

📊 6,311 reviews | 5 hotels
📅 2011-05-29 → 2026-03-08

Reviews per hotel:
hotel
Banff Caribou Lodge & Spa    1887
Banff Ptarmigan Inn          1331
Mount Royal Hotel            1186
The Rundlestone Lodge        1016
Banff Inn                     891

Detailed ratings coverage:
  rating_rooms: 1,818 / 6,311 (29%)
  rating_service: 1,829 / 6,311 (29%)
  rating_location: 1,828 / 6,311 (29%)

Trip type breakdown:
trip_type
Vacation    1152
Business      71

Travel group breakdown:
travel_group
Couple     512
Family     420
Friends    180
Solo        76


,place_id,hotel,reviewer,reviewer_id,stars,date,year_month,year,month,text,...,reviewer_reviews,reviewer_photos,rating_rooms,rating_service,rating_location,trip_type,travel_group,highlights,owner_response,source
0,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,Steve Davenport,117239732157986962145,4,2026-02-21 19:38:39,2026-02-01,2026,2,Solid four out of five and overall we really e...,...,16,0,4.0,4.0,4.0,Vacation,Family,"Quiet, Great value",NaN,google
1,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,Yicheng Zhang,106668418751164756702,5,2026-02-13 21:29:21,2026-02-01,2026,2,"Amazing hotel! Great room and service, great l...",...,901,2242,5.0,5.0,5.0,NaN,NaN,NaN,"Hi Yicheng,\n\nThank you so much for your wond...",google
2,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,Parm Mundi,112077600729292483220,1,2026-02-01 04:25:11,2026-02-01,2026,2,Our experience at this hotel was not great at ...,...,3,0,1.0,1.0,3.0,Vacation,Family,NaN,"Hi Parm,\n\nThank you for taking the time to s...",google
3,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,Meet Patel,105551934500342185656,5,2026-01-30 23:16:42,2026-01-01,2026,1,"Best location, best staff, always super clean....",...,19,28,5.0,5.0,5.0,Business,Friends,"Luxury, Great view, Quiet","Hi Meet,\n\nThank you so much for your wonderf...",google
4,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,sarah kangas-hanes,104398928022987670439,1,2026-01-23 03:04:10,2026-01-01,2026,1,NaN,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Thank you for taking the time to leave a ratin...,google
5,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,rick murray,105657398660827708456,1,2026-01-04 15:44:49,2026-01-01,2026,1,A friend of mine and I decided to book our sta...,...,17,3,4.0,2.0,4.0,NaN,NaN,NaN,Thank you for taking the time to share your fe...,google
6,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,Cat Russell,105551534801372491971,2,2025-12-01 16:37:41,2025-12-01,2025,12,The man at the desk let me know in no uncertai...,...,159,145,1.0,1.0,5.0,NaN,NaN,NaN,Thank you for taking the time to share your fe...,google
8,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,chase patenaude,105294815411759731677,1,2025-12-27 00:35:34,2025-12-01,2025,12,Don't stay here,...,8,62,NaN,NaN,NaN,NaN,NaN,NaN,Thank you for taking the time to share your fe...,google
9,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,Yazmin Garcia,105513117570571961948,5,2025-12-17 20:02:06,2025-12-01,2025,12,NaN,...,8,60,5.0,5.0,5.0,NaN,NaN,NaN,Thanks so much for taking the time to share yo...,google
10,ChIJXfkQg03KcFMRIX8QSHEhTfY,Banff Inn,Tim Fauser,114594590018030892731,4,2025-12-17 15:14:16,2025-12-01,2025,12,Room was quite nice. Atmosphere is not as fanc...,...,183,109,5.0,4.0,5.0,Vacation,Family,Quiet,Thank you for taking the time to share your fe...,google


## 2. Quick Summary Stats

In [28]:
# ── Summary stats ─────────────────────────────────────────────
summary = df.groupby('hotel').agg(
    total_reviews  = ('stars', 'count'),
    avg_rating     = ('stars', 'mean'),
    median_rating  = ('stars', 'median'),
    pct_5star      = ('stars', lambda x: (x == 5).mean() * 100),
    pct_1star      = ('stars', lambda x: (x == 1).mean() * 100),
    owner_response_rate = ('owner_response', lambda x: x.notna().mean() * 100),
    first_review   = ('date', 'min'),
    last_review    = ('date', 'max'),
).round(2)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(summary)


summary = df.groupby('hotel').agg(
    avg_rating     = ('stars', 'mean'),
    pct_5star      = ('stars', lambda x: (x == 5).mean() * 100),
    pct_1star      = ('stars', lambda x: (x == 1).mean() * 100),
    owner_response_rate = ('owner_response', lambda x: x.notna().mean() * 100),
).round(2)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(summary)




,total_reviews,avg_rating,median_rating,pct_5star,pct_1star,owner_response_rate,first_review,last_review
hotel,,,,,,,,
Banff Caribou Lodge & Spa,1887,3.98,4.0,42.77,7.37,0.16,2013-07-01 14:50:30,2026-03-08 15:42:54
Banff Inn,891,4.10,4.0,47.81,6.96,5.27,2016-01-23 18:56:49,2026-02-21 19:38:39
Banff Ptarmigan Inn,1331,4.06,4.0,45.23,5.33,0.00,2016-03-15 02:30:56,2026-03-04 10:14:16
Mount Royal Hotel,1186,4.27,5.0,57.08,4.89,24.96,2011-05-29 23:09:42,2026-03-01 22:21:22
The Rundlestone Lodge,1016,4.11,4.0,48.43,6.89,29.63,2014-06-04 17:40:33,2026-03-01 04:24:07


,avg_rating,pct_5star,pct_1star,owner_response_rate
hotel,,,,
Banff Caribou Lodge & Spa,3.98,42.77,7.37,0.16
Banff Inn,4.10,47.81,6.96,5.27
Banff Ptarmigan Inn,4.06,45.23,5.33,0.00
Mount Royal Hotel,4.27,57.08,4.89,24.96
The Rundlestone Lodge,4.11,48.43,6.89,29.63


In [29]:
box_output = widgets.Output()

min_date = df['date'].min().date()
max_date = df['date'].max().date()

date_slider = widgets.SelectionRangeSlider(
    options=pd.date_range(min_date, max_date, freq='MS').tolist(),
    index=(0, len(pd.date_range(min_date, max_date, freq='MS')) - 1),
    description='Date range:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='700px')
)

box_dd_left = widgets.Dropdown(
    options=sorted(df['hotel'].unique().tolist()),
    value=sorted(df['hotel'].unique().tolist())[0],
    description='Hotel A:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
box_dd_right = widgets.Dropdown(
    options=sorted(df['hotel'].unique().tolist()),
    value=sorted(df['hotel'].unique().tolist())[1] if len(df['hotel'].unique()) > 1 else sorted(df['hotel'].unique().tolist())[0],
    description='Hotel B:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

def plot_box(_):
    box_output.clear_output(wait=True)
    with box_output:
        h_a   = box_dd_left.value
        h_b   = box_dd_right.value
        start = pd.Timestamp(date_slider.value[0])
        end   = pd.Timestamp(date_slider.value[1])

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[f'<b>{h_a}</b>', f'<b>{h_b}</b>'],
            shared_yaxes=True
        )

        for col, (hotel, color) in enumerate(
            [(h_a, '#4C72B0'), (h_b, '#DD8452')], start=1
        ):
            dff = df[
                (df['hotel'] == hotel) &
                (df['date'] >= start) &
                (df['date'] <= end)
            ]

            counts = dff['stars'].value_counts().sort_index().reindex([1,2,3,4,5], fill_value=0)
            pct    = (counts / counts.sum() * 100).round(1)
            n      = len(dff)
            avg    = dff['stars'].mean()

            fig.add_trace(go.Bar(
                x=counts.index.astype(str) + '★',
                y=pct.values,
                marker_color=color,
                opacity=0.85,
                text=pct.astype(str) + '%',
                textposition='outside',
                hovertemplate='%{x}<br>%{y:.1f}%<extra></extra>',
                name=hotel,
                showlegend=False,
            ), row=1, col=col)

            fig.add_annotation(
                text=f'n={n:,} | mean={avg:.2f}★',
                xref='paper', yref='paper',
                x=0.25 if col == 1 else 0.75,
                y=-0.15,
                showarrow=False,
                font=dict(size=11, color='gray')
            )

        fig.update_yaxes(
            title_text='% of Reviews',
            gridcolor='rgba(200,200,200,0.5)',
            showline=True, linecolor='black',
            ticks='outside', range=[0, 85],
        )
        fig.update_xaxes(
            showline=True, linecolor='black',
            ticks='outside',
        )
        fig.update_layout(
            title=dict(text='<b>Rating Distribution Comparison</b>', font=dict(size=16)),
            plot_bgcolor='white',
            paper_bgcolor='white',
            height=460,
            margin=dict(l=60, r=40, t=80, b=60),
        )
        fig.show()

for w in [box_dd_left, box_dd_right, date_slider]:
    w.observe(plot_box, names='value')

display(
    widgets.HTML('<b>Compare rating distributions:</b>'),
    widgets.HBox([box_dd_left, box_dd_right]),
    date_slider,
    box_output
)
plot_box(None)


HTML(value='<b>Compare rating distributions:</b>')

SelectionRangeSlider(description='Date range:', index=(0, 177), layout=Layout(width='700px'), options=(Timesta…

Output()

In [30]:
all5_output = widgets.Output()

all5_date_slider = widgets.SelectionRangeSlider(
    options=pd.date_range(min_date, max_date, freq='MS').tolist(),
    index=(0, len(pd.date_range(min_date, max_date, freq='MS')) - 1),
    description='Date range:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='700px')
)

def plot_all5(_):
    all5_output.clear_output(wait=True)
    with all5_output:
        start  = pd.Timestamp(all5_date_slider.value[0])
        end    = pd.Timestamp(all5_date_slider.value[1])
        hotels = sorted(df['hotel'].unique().tolist())
        n_hotels = len(hotels)

        colors = px.colors.qualitative.D3

        fig = make_subplots(
            rows=1, cols=n_hotels,
            subplot_titles=[f'<b>{h}</b>' for h in hotels],
            shared_yaxes=True
        )

        for col, hotel in enumerate(hotels, start=1):
            dff = df[
                (df['hotel'] == hotel) &
                (df['date'] >= start) &
                (df['date'] <= end)
            ]

            counts = (dff['stars'].value_counts()
                      .sort_index()
                      .reindex([1,2,3,4,5], fill_value=0))
            pct    = (counts / counts.sum() * 100).round(1)
            n      = len(dff)
            avg    = dff['stars'].mean()
            color  = colors[(col-1) % len(colors)]

            fig.add_trace(go.Bar(
                x=counts.index.astype(str) + '★',
                y=pct.values,
                marker_color=color,
                opacity=0.85,
                text=pct.astype(str) + '%',
                textposition='outside',
                name=hotel,
                showlegend=False,
                hovertemplate='%{x}<br>%{y:.1f}%<extra></extra>',
            ), row=1, col=col)

            # n= and mean annotation below each panel
            fig.add_annotation(
                text=f'n={n:,}<br>mean={avg:.2f}★',
                xref='paper', yref='paper',
                x=(col - 0.5) / n_hotels,
                y=-0.2,
                showarrow=False,
                font=dict(size=10, color='gray'),
                align='center'
            )

        fig.update_yaxes(
            gridcolor='rgba(200,200,200,0.5)',
            showgrid=True,
            showline=True, linecolor='black',
            ticks='outside',
            range=[0, 90],
        )
        fig.update_yaxes(
            title_text='% of Reviews',
            col=1
        )
        fig.update_xaxes(
            showline=True, linecolor='black',
            ticks='outside',
        )
        fig.update_layout(
            title=dict(
                text='<b>Rating Distribution — All Hotels</b>',
                font=dict(size=16)
            ),
            plot_bgcolor='white',
            paper_bgcolor='white',
            height=460,
            width=300 * n_hotels,
            margin=dict(l=60, r=40, t=80, b=80),
        )
        fig.show()

all5_date_slider.observe(plot_all5, names='value')

display(
    widgets.HTML('<b>Rating Distribution — All Hotels:</b>'),
    all5_date_slider,
    all5_output
)
plot_all5(None)

HTML(value='<b>Rating Distribution — All Hotels:</b>')

SelectionRangeSlider(description='Date range:', index=(0, 177), layout=Layout(width='700px'), options=(Timesta…

Output()

## 3. 📈 Interactive Rating Trend — Selectable Time Window

In [31]:
# ── Widget controls ───────────────────────────────────────────
trend_hotels = sorted(df['hotel'].unique().tolist())

trend_dd_left = widgets.Dropdown(
    options=trend_hotels,
    value=trend_hotels[0],
    description='Hotel A:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
trend_dd_right = widgets.Dropdown(
    options=trend_hotels,
    value=trend_hotels[1] if len(trend_hotels) > 1 else trend_hotels[0],
    description='Hotel B:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
trend_granularity = widgets.Dropdown(
    options=[('Monthly', 'ME'), ('Quarterly', 'QE'), ('Yearly', 'YE')],
    value='QE',
    description='Granularity:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

min_date = df['date'].min().date()
max_date = df['date'].max().date()

trend_date_range = widgets.SelectionRangeSlider(
    options=pd.date_range(min_date, max_date, freq='MS').tolist(),
    index=(0, len(pd.date_range(min_date, max_date, freq='MS')) - 1),
    description='Date window:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='700px')
)
trend_show_volume = widgets.Checkbox(
    value=True,
    description='Show review volume',
    layout=widgets.Layout(width='200px')
)
trend_show_rolling = widgets.Checkbox(
    value=True,
    description='Show 3-period weighted avg',
    layout=widgets.Layout(width='230px')
)

trend_output = widgets.Output()

# ── Helpers ───────────────────────────────────────────────────
def weighted_rolling(agg, window=3):
    def _calc(idx):
        s = max(0, idx - window + 1)
        w = agg.iloc[s:idx+1]
        return (w['avg_rating'] * w['review_count']).sum() / w['review_count'].sum()
    return [_calc(i) for i in range(len(agg))]

def get_agg(hotel, granularity, start, end):
    mask = (
        (df['hotel'] == hotel) &
        (df['date'] >= start) &
        (df['date'] <= end)
    )
    dff = df[mask].set_index('date')
    agg = dff['stars'].resample(granularity).agg(['mean','count']).reset_index()
    agg.columns = ['date', 'avg_rating', 'review_count']
    agg.dropna(subset=['avg_rating'], inplace=True)
    return agg

# ── Plot ──────────────────────────────────────────────────────
def plot_trend_comparison(_):
    trend_output.clear_output(wait=True)
    with trend_output:
        h_a       = trend_dd_left.value
        h_b       = trend_dd_right.value
        gran      = trend_granularity.value
        start     = pd.Timestamp(trend_date_range.value[0])
        end       = pd.Timestamp(trend_date_range.value[1])
        vol       = trend_show_volume.value
        rolling   = trend_show_rolling.value
        gran_label= {'ME': 'Monthly', 'QE': 'Quarterly', 'YE': 'Yearly'}[gran]

        colors = {h_a: '#4C72B0', h_b: '#DD8452'}
        roll_colors = {h_a: '#1a4a8a', h_b: '#a04010'}

        specs = [[{'secondary_y': True}]] if vol else [[{}]]
        fig = make_subplots(specs=specs)

        for hotel in [h_a, h_b]:
            agg = get_agg(hotel, gran, start, end)
            if agg.empty:
                continue

            # volume bars
            if vol:
                fig.add_trace(go.Bar(
                    x=agg['date'], y=agg['review_count'],
                    name=f'{hotel} — volume',
                    marker_color=colors[hotel],
                    opacity=0.15,
                    hovertemplate=f'{hotel}<br>%{{x|%b %Y}}<br>Reviews: %{{y}}<extra></extra>'
                ), secondary_y=True)

            # avg rating scatter
            fig.add_trace(go.Scatter(
                x=agg['date'], y=agg['avg_rating'],
                mode='markers',
                name=f'{hotel} — avg',
                marker=dict(color=colors[hotel], size=7),
                hovertemplate=f'{hotel}<br>%{{x|%b %Y}}<br>Avg: %{{y:.2f}}★<extra></extra>'
            ), secondary_y=False)

            # weighted rolling average
            if rolling:
                agg['rolling'] = weighted_rolling(agg, window=3)
                fig.add_trace(go.Scatter(
                    x=agg['date'], y=agg['rolling'],
                    mode='lines',
                    name=f'{hotel} — 3-period avg',
                    line=dict(color=roll_colors[hotel], width=2.5),
                    hovertemplate=f'{hotel}<br>%{{x|%b %Y}}<br>Rolling: %{{y:.2f}}★<extra></extra>'
                ), secondary_y=False)

        fig.update_layout(
            title=dict(
                text=f'<b>Rating Trend Comparison</b> — {gran_label}',
                font=dict(size=16)
            ),
            xaxis=dict(
                title='Date', showgrid=True,
                gridcolor='rgba(200,200,200,0.3)'
            ),
            yaxis=dict(
                title='Avg Star Rating',
                range=[1, 5.5],
                showgrid=True,
                gridcolor='rgba(200,200,200,0.3)'
            ),
            legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
            plot_bgcolor='white',
            paper_bgcolor='white',
            hovermode='x unified',
            barmode='group',
            height=500,
        )
        if vol:
            fig.update_yaxes(title_text='Review Count', secondary_y=True, showgrid=False)

        fig.show()

# ── Wire up & render ──────────────────────────────────────────
for w in [trend_dd_left, trend_dd_right, trend_granularity,
          trend_date_range, trend_show_volume, trend_show_rolling]:
    w.observe(plot_trend_comparison, names='value')

controls = widgets.VBox([
    widgets.HBox([trend_dd_left, trend_dd_right, trend_granularity]),
    trend_date_range,
    widgets.HBox([trend_show_volume, trend_show_rolling]),
])

display(controls, trend_output)
plot_trend_comparison(None)

Output()

# NLP of Reviews

In [32]:
nlp = spacy.load("en_core_web_sm")
analyzer = SentimentIntensityAnalyzer()

# ── Hotel-specific stopwords ──────────────────────────────────
STOPWORDS = {
    # pronouns / generic
    'you', 'they', 'we', 'it', 'i', 'he', 'she', 'that', 'this', 'these',
    'those', 'there', 'here', 'thing', 'everything', 'anything', 'nothing',
    'someone', 'one', 'way', 'lot', 'time', 'anyone', 'everyone', 'something',
    'bit', 'kind', 'sort', 'type', 'part', 'side', 'end', 'top', 'back',
    'front', 'night', 'day', 'year', 'week', 'visit', 'experience', 'spot',
    # hotel domain noise
    'hotel', 'stay', 'room', 'place', 'trip', 'town', 'group', 'friend',
    'people', 'banff', 'canmore', 'property', 'guest', 'management'
}

MIN_FREQ = 3

records = []
reviews = df.dropna(subset=['text']).copy()

for _, row in reviews.iterrows():
    doc = nlp(row['text'])
    for sent in doc.sents:
        compound = analyzer.polarity_scores(sent.text)['compound']
        label = 'positive' if compound >= 0.05 else ('negative' if compound <= -0.05 else 'neutral')

        for chunk in sent.noun_chunks:
            root = chunk.root
            if root.pos_ != 'NOUN':
                continue
            lemma = root.lemma_.lower()
            if lemma in STOPWORDS or len(lemma) <= 2:
                continue

            chunk_words = [
                t for t in chunk
                if not t.is_stop and not t.is_punct
                and t.pos_ in ('NOUN', 'ADJ')
            ]
            aspect = ' '.join(t.lemma_.lower() for t in chunk_words[:2]) if len(chunk_words) >= 2 else lemma

            records.append({
                'aspect':         aspect,
                'sentiment':      label,
                'compound':       compound,
                'stars':          row['stars'],
                'hotel':          row['hotel'],
                'reviewer':       row['reviewer'],
                'date':           row['date'],
                # reviewer features
                'is_local_guide':    row['is_local_guide'],
                'reviewer_reviews':  row['reviewer_reviews'],
                'reviewer_photos':   row['reviewer_photos'],
                # detailed ratings
                'rating_rooms':      row['rating_rooms'],
                'rating_service':    row['rating_service'],
                'rating_location':   row['rating_location'],
                # segmentation
                'trip_type':         row['trip_type'],
                'travel_group':      row['travel_group'],
                'highlights':        row['highlights'],
                # text features
                'text_length':       len(str(row['text'])),
                'has_owner_response': int(pd.notna(row['owner_response'])),
                'source':            row['source'],
                'sentence':          sent.text.strip()
            })

aspect_df = pd.DataFrame(records)

# ── Min frequency filter ──────────────────────────────────────
freq = aspect_df['aspect'].value_counts()
aspect_df = aspect_df[aspect_df['aspect'].isin(freq[freq >= MIN_FREQ].index)]

print(f"✅ Extracted {len(aspect_df):,} aspect-sentiment pairs")
print(f"\nTop 200 aspects — review before building CATEGORY_MAP:\n")

top200 = aspect_df['aspect'].value_counts().head(500)
#print(top200.to_string())

CATEGORY_MAP = {
    'Rooms': [
        'bed', 'bathroom', 'shower', 'toilet', 'pillow', 'sheet', 'towel',
        'window', 'balcony', 'suite', 'wall', 'floor', 'ceiling', 'carpet',
        'fridge', 'microwave', 'air conditioning', 'air conditioner', 'heat',
        'thermostat', 'fan', 'noise', 'wifi', 'tv', 'safe', 'desk',
        'comfortable bed', 'comfy bed', 'queen bed', 'king bed', 'king suite',
        'mini fridge', 'coffee maker', 'coffee machine', 'kettle', 'robe',
        'toiletry', 'shampoo', 'soap', 'lotion', 'conditioner', 'body wash',
        'mattress', 'duvet', 'blanket', 'linen', 'bed sheet', 'bedsheet',
        'furniture', 'fireplace', 'bathtub', 'jetted tub', 'jacuzzi tub',
        'sink', 'washroom', 'stain', 'dust', 'cleanliness', 'decor',
        'renovation', 'bedroom', 'loft', 'size', 'good size', 'lighting',
        'curtain', 'outlet', 'plug', 'smell',
    ],
    'Service': [
        'staff', 'service', 'desk', 'desk staff', 'front desk', 'manager',
        'receptionist', 'reception', 'reception staff', 'housekeeping',
        'housekeeper', 'employee', 'team', 'server', 'customer service',
        'great service', 'good service', 'excellent service', 'amazing service',
        'friendly staff', 'great staff', 'nice staff', 'helpful staff',
        'hotel staff', 'staff member', 'lovely staff', 'wonderful staff',
        'amazing staff', 'clean friendly', 'friendly service',
        'friendly helpful', 'excellent customer', 'great customer',
        'poor customer', 'hospitality', 'checkin', 'checkout',
        'early check', 'reservation', 'booking', 'complaint', 'apology',
        'response', 'communication', 'attitude', 'maintenance',
    ],
    'Location': [
        'location', 'downtown', 'great location', 'good location',
        'perfect location', 'excellent location', 'amazing location',
        'fantastic location', 'convenient location', 'nice location',
        'central location', 'walking distance', 'minute walk', 'min walk',
        'short walk', 'distance', 'street', 'main street', 'main strip',
        'main road', 'downtown area', 'downtown core', 'town centre',
        'centre', 'center', 'area', 'shop', 'shopping', 'parking',
        'free parking', 'underground parking', 'street parking',
        'parking garage', 'parking space', 'underground parkade',
        'parkade', 'free bus', 'bus', 'bus stop', 'shuttle', 'free shuttle',
        'ski bus', 'access', 'easy access', 'hike', 'hiking', 'skiing', 'ski','heart','middle'
    ],
    'Amenities': [
        'hot tub', 'pool', 'spa', 'sauna', 'gym', 'hot pool', 'jacuzzi',
        'rooftop hot', 'rooftop', 'hottub', 'tub', 'swimming pool',
        'pool area', 'nice pool', 'nice hot', 'big hot', 'jetted tub',
        'lounge', 'bar', 'lobby', 'elevator', 'laundry', 'atrium',
        'facility', 'amenity', 'good amenity', 'great amenity', 'restaurant',
        'hotel restaurant', 'good restaurant', 'great restaurant',
        'room service', 'massage', 'fireplace', 'coffee', 'free coffee',
        'ice machine', 'ice', 'kitchen',
    ],
    'Food & Breakfast': [
        'breakfast', 'food', 'dinner', 'meal', 'menu', 'buffet',
        'free breakfast', 'complimentary breakfast', 'continental breakfast',
        'breakfast buffet', 'buffet breakfast', 'great breakfast',
        'nice breakfast', 'good breakfast', 'excellent breakfast',
        'great food', 'good food', 'omelette', 'bacon', 'egg',
        'fruit', 'cereal', 'drink', 'wine', 'beer', 'pizza',
    ],
    'Value': [
        'price', 'money', 'cost', 'rate', 'value', 'great value',
        'good value', 'good price', 'great price', 'reasonable price',
        'decent price', 'high price', 'price point', 'budget',
        'refund', 'deposit', 'charge', 'fee', 'discount', 'upgrade',
        'credit card', 'dollar', 'bill', 'booking',
    ],
}

def map_category(aspect):
    for category, keywords in CATEGORY_MAP.items():
        if aspect in keywords:
            return category
    return None

aspect_df['category'] = aspect_df['aspect'].apply(map_category)
categorized = aspect_df.dropna(subset=['category'])

print(f"✅ Categorized {len(categorized):,} of {len(aspect_df):,} mentions ({len(categorized)/len(aspect_df)*100:.0f}%)")
print(f"\nMentions per category:")
print(categorized.groupby('category').size().sort_values(ascending=False).to_string())

✅ Extracted 17,395 aspect-sentiment pairs

Top 200 aspects — review before building CATEGORY_MAP:

✅ Categorized 9,614 of 17,395 mentions (55%)

Mentions per category:
category
Rooms               2391
Location            2177
Service             1881
Amenities           1856
Food & Breakfast     733
Value                576


In [33]:
# ── Aspect mentions over time ─────────────────────────────────
hotels = sorted(aspect_df['hotel'].unique().tolist())

amt_dd_left = widgets.Dropdown(
    options=hotels,
    value=hotels[0],
    description='Hotel A:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
amt_dd_right = widgets.Dropdown(
    options=hotels,
    value=hotels[1] if len(hotels) > 1 else hotels[0],
    description='Hotel B:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
amt_granularity = widgets.Dropdown(
    options=[('Monthly', 'ME'), ('Quarterly', 'QE'), ('Yearly', 'YE')],
    value='QE',
    description='Granularity:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)
amt_date_range = widgets.SelectionRangeSlider(
    options=pd.date_range(min_date, max_date, freq='MS').tolist(),
    index=(0, len(pd.date_range(min_date, max_date, freq='MS')) - 1),
    description='Date window:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='700px')
)
amt_output = widgets.Output()

def get_top_aspects(hotel, start, end, n=10):
    dff = aspect_df[
        (aspect_df['hotel'] == hotel) &
        (aspect_df['date'] >= start) &
        (aspect_df['date'] <= end)
    ]
    return dff['aspect'].value_counts().head(n).index.tolist()

def plot_aspect_time(hotel, top_aspects, start, end, granularity, row, fig):
    dff = aspect_df[
        (aspect_df['hotel'] == hotel) &
        (aspect_df['date'] >= start) &
        (aspect_df['date'] <= end) &
        (aspect_df['aspect'].isin(top_aspects))
    ].copy()

    if dff.empty:
        return

    dff = dff.set_index('date')
    colors = px.colors.qualitative.D3

    for i, aspect in enumerate(top_aspects):
        asp_df = dff[dff['aspect'] == aspect]
        if asp_df.empty:
            continue

        agg = asp_df.resample(granularity).agg(
            mentions  = ('compound', 'count'),
            positivity = ('compound', lambda x: (x > 0.05).mean() * 100)
        ).reset_index()
        agg.columns = ['date', 'mentions', 'positivity']
        agg = agg[agg['mentions'] > 0]

        color = colors[i % len(colors)]

        # mentions bars — primary y
        fig.add_trace(go.Bar(
            x=agg['date'], y=agg['mentions'],
            name=aspect,
            marker_color=color,
            opacity=0.6,
            legendgroup=aspect,
            showlegend=(row == 1),
            hovertemplate=f'{aspect}<br>%{{x|%b %Y}}<br>Mentions: %{{y}}<extra></extra>'
        ), row=row, col=1, secondary_y=False)

        # positivity line — secondary y
        fig.add_trace(go.Scatter(
            x=agg['date'], y=agg['positivity'],
            name=f'{aspect} positivity',
            mode='lines+markers',
            line=dict(color=color, width=1.5, dash='dot'),
            marker=dict(size=4),
            legendgroup=aspect,
            showlegend=False,
            hovertemplate=f'{aspect}<br>%{{x|%b %Y}}<br>Positive: %{{y:.0f}}%<extra></extra>'
        ), row=row, col=1, secondary_y=True)

def plot_amt_comparison(_):
    amt_output.clear_output(wait=True)
    with amt_output:
        h_a   = amt_dd_left.value
        h_b   = amt_dd_right.value
        gran  = amt_granularity.value
        start = pd.Timestamp(amt_date_range.value[0])
        end   = pd.Timestamp(amt_date_range.value[1])
        gran_label = {'ME': 'Monthly', 'QE': 'Quarterly', 'YE': 'Yearly'}[gran]

        # ── Global top 10 across both hotels ──────────────────
        both = aspect_df[
            (aspect_df['hotel'].isin([h_a, h_b])) &
            (aspect_df['date'] >= start) &
            (aspect_df['date'] <= end)
        ]
        top10 = both['aspect'].value_counts().head(10).index.tolist()

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[f'<b>{h_a}</b>', f'<b>{h_b}</b>'],
            shared_yaxes=True
        )

        colors = px.colors.qualitative.D3
        # track which aspects have been added to legend
        legend_added = set()

        def smooth(series, window=3):
            return [
                series.iloc[max(0, i-window+1):i+1].mean()
                for i in range(len(series))
            ]

        for col, hotel in enumerate([h_a, h_b], start=1):
            dff = aspect_df[
                (aspect_df['hotel'] == hotel) &
                (aspect_df['date'] >= start) &
                (aspect_df['date'] <= end) &
                (aspect_df['aspect'].isin(top10))
            ].copy().set_index('date')

            for i, aspect in enumerate(top10):
                asp_df = dff[dff['aspect'] == aspect]
                color  = colors[i % len(colors)]
                show_legend = aspect not in legend_added

                if asp_df.empty:
                    # add invisible trace to keep legend entry
                    if show_legend:
                        fig.add_trace(go.Scatter(
                            x=[], y=[],
                            name=aspect,
                            mode='lines',
                            line=dict(color=color, width=2.5),
                            legendgroup=aspect,
                            showlegend=True,
                        ), row=1, col=col)
                        legend_added.add(aspect)
                    continue

                agg = (
                    asp_df.resample(gran)['compound']
                    .agg(positivity=lambda x: (x > 0.05).mean() * 100)
                    .reset_index()
                )
                agg.columns = ['date', 'positivity']
                agg = agg[agg['positivity'].notna()]

                if len(agg) < 2:
                    continue

                agg['smoothed'] = smooth(agg['positivity'])

                # raw dots
                fig.add_trace(go.Scatter(
                    x=agg['date'], y=agg['positivity'],
                    mode='markers',
                    marker=dict(color=color, size=5, opacity=0.3),
                    legendgroup=aspect,
                    showlegend=False,
                    hovertemplate=f'{aspect}<br>%{{x|%b %Y}}<br>Raw: %{{y:.0f}}%<extra></extra>'
                ), row=1, col=col)

                # smoothed line
                fig.add_trace(go.Scatter(
                    x=agg['date'], y=agg['smoothed'],
                    mode='lines',
                    name=aspect,
                    line=dict(color=color, width=2.5),
                    legendgroup=aspect,
                    showlegend=show_legend,
                    hovertemplate=f'{aspect}<br>%{{x|%b %Y}}<br>Smoothed: %{{y:.0f}}%<extra></extra>'
                ), row=1, col=col)

                legend_added.add(aspect)

        fig.update_yaxes(
            title_text='% Positive Mentions',
            range=[0, 105],
            showgrid=True,
            gridcolor='rgba(200,200,200,0.4)',
            ticksuffix='%',
            showline=True, linecolor='black',
            row=1, col=1
        )
        fig.update_xaxes(
            showgrid=True,
            gridcolor='rgba(200,200,200,0.2)',
            showline=True, linecolor='black',
        )
        fig.add_hline(
            y=50, line_dash='dash', line_color='gray',
            line_width=1, opacity=0.5,
            annotation_text='50%', annotation_position='left'
        )
        fig.update_layout(
            title=dict(
                text=(f'<b>Aspect Positivity Over Time</b> — {gran_label}<br>'
                      f'<sup>3-period smoothed | dots = raw | '
                      f'click legend to toggle</sup>'),
                font=dict(size=14)
            ),
            plot_bgcolor='white',
            paper_bgcolor='white',
            height=480,
            hovermode='x unified',
            legend=dict(
                title='Aspect',
                orientation='v',
                yanchor='top', y=1,
                xanchor='left', x=1.02
            ),
        )
        fig.show()

for w in [amt_dd_left, amt_dd_right, amt_granularity, amt_date_range]:
    w.observe(plot_amt_comparison, names='value')

display(
    widgets.HTML('<b>Aspect Positivity Over Time:</b>'),
    widgets.HBox([amt_dd_left, amt_dd_right, amt_granularity]),
    amt_date_range,
    amt_output
)
plot_amt_comparison(None)

HTML(value='<b>Aspect Positivity Over Time:</b>')

SelectionRangeSlider(description='Date window:', index=(0, 177), layout=Layout(width='700px'), options=(Timest…

Output()

In [34]:
# ── Aspect sentiment side-by-side comparison ─────────────────
hotels = sorted(aspect_df['hotel'].unique().tolist())

asp_dd_left = widgets.Dropdown(
    options=hotels,
    value=hotels[0],
    description='Hotel A:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
asp_dd_right = widgets.Dropdown(
    options=hotels,
    value=hotels[1] if len(hotels) > 1 else hotels[0],
    description='Hotel B:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

min_date = df['date'].min().date()
max_date = df['date'].max().date()

asp_date_range = widgets.SelectionRangeSlider(
    options=pd.date_range(min_date, max_date, freq='MS').tolist(),
    index=(0, len(pd.date_range(min_date, max_date, freq='MS')) - 1),
    description='Date window:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='700px')
)

TOP_N       = 15
asp_output  = widgets.Output()
color_map   = {'positive': '#2ca02c', 'neutral': '#aec7e8', 'negative': '#d62728'}

def get_asp_counts(hotel, start, end):
    mask = (
        (aspect_df['hotel'] == hotel) &
        (aspect_df['date'] >= start) &
        (aspect_df['date'] <= end)
    ) if 'date' in aspect_df.columns else (aspect_df['hotel'] == hotel)

    dff      = aspect_df[mask]
    top      = dff['aspect'].value_counts().head(TOP_N).index.tolist()
    filtered = dff[dff['aspect'].isin(top)]

    counts   = (
        filtered.groupby(['aspect', 'sentiment'])
        .size().reset_index(name='count')
    )
    total    = counts.groupby('aspect')['count'].transform('sum')
    counts['pct']   = (counts['count'] / total * 100).round(1)
    counts['total'] = total
    return counts

def get_asp_heatmap(hotel, start, end):
    mask = (
        (aspect_df['hotel'] == hotel) &
        (aspect_df['date'] >= start) &
        (aspect_df['date'] <= end)
    ) if 'date' in aspect_df.columns else (aspect_df['hotel'] == hotel)

    dff      = aspect_df[mask]
    top      = dff['aspect'].value_counts().head(TOP_N).index.tolist()
    filtered = dff[dff['aspect'].isin(top)]

    return (
        filtered.groupby(['aspect', 'stars'])['compound']
        .mean().reset_index()
        .pivot(index='aspect', columns='stars', values='compound')
        .fillna(0)
    )

def plot_asp_comparison(_):
    asp_output.clear_output(wait=True)
    with asp_output:
        h_a   = asp_dd_left.value
        h_b   = asp_dd_right.value
        start = pd.Timestamp(asp_date_range.value[0])
        end   = pd.Timestamp(asp_date_range.value[1])

        # ── Bar charts ────────────────────────────────────────
        fig_bar = make_subplots(
            rows=1, cols=2,
            subplot_titles=[f'<b>{h_a}</b>', f'<b>{h_b}</b>'],
        )

        for col, hotel in enumerate([h_a, h_b], start=1):
            counts     = get_asp_counts(hotel, start, end)
            sort_order = (
                counts[counts['sentiment'] == 'positive']
                .sort_values('pct', ascending=True)['aspect'].tolist()
            )
            for sentiment, color in color_map.items():
                s = counts[counts['sentiment'] == sentiment]
                fig_bar.add_trace(go.Bar(
                    x=s['pct'], y=s['aspect'],
                    orientation='h',
                    name=sentiment,
                    marker_color=color,
                    text=s['pct'].astype(str) + '%',
                    textposition='inside',
                    showlegend=(col == 1),
                    hovertemplate=f'{hotel}<br>%{{y}}<br>%{{x:.1f}}%<extra></extra>'
                ), row=1, col=col)

            # n= annotations
            for aspect in counts['aspect'].unique():
                n = counts[counts['aspect'] == aspect]['total'].iloc[0]
                fig_bar.add_annotation(
                    x=103, y=aspect,
                    text=f'n={n}',
                    showarrow=False,
                    font=dict(size=9, color='gray'),
                    xanchor='left',
                    row=1, col=col
                )

        fig_bar.update_layout(
            barmode='stack',
            title='<b>Aspect Sentiment Comparison</b>',
            height=560,
            plot_bgcolor='white',
            legend_title='Sentiment',
        )
        fig_bar.update_xaxes(range=[0, 100], title_text='% of mentions')
        fig_bar.show()

        # ── Heatmaps ──────────────────────────────────────────
        fig_heat = make_subplots(
            rows=1, cols=2,
            subplot_titles=[f'<b>{h_a}</b>', f'<b>{h_b}</b>'],
        )

        for col, hotel in enumerate([h_a, h_b], start=1):
            hd = get_asp_heatmap(hotel, start, end)
            fig_heat.add_trace(go.Heatmap(
                z=hd.values,
                x=[f'{s}⭐' for s in hd.columns],
                y=hd.index.tolist(),
                colorscale=[
                    [0.0, '#d62728'],
                    [0.5, '#f7f7f7'],
                    [1.0, '#2ca02c']
                ],
                zmid=0,
                text=hd.values.round(2),
                texttemplate='%{text}',
                showscale=(col == 1),
                hovertemplate=f'{hotel}<br>%{{y}}<br>Stars: %{{x}}<br>Score: %{{z:.2f}}<extra></extra>',
            ), row=1, col=col)

        fig_heat.update_layout(
            title='<b>Aspect Sentiment by Star Rating — Comparison</b>',
            height=560,
            plot_bgcolor='white',
        )
        fig_heat.show()

# ── Wire up & render ──────────────────────────────────────────
for w in [asp_dd_left, asp_dd_right, asp_date_range]:
    w.observe(plot_asp_comparison, names='value')

display(
    widgets.HTML('<b>Compare aspect sentiment across hotels:</b>'),
    widgets.HBox([asp_dd_left, asp_dd_right]),
    asp_date_range,
    asp_output
)
plot_asp_comparison(None)

HTML(value='<b>Compare aspect sentiment across hotels:</b>')

SelectionRangeSlider(description='Date window:', index=(0, 177), layout=Layout(width='700px'), options=(Timest…

Output()

In [35]:
# ── Category sentiment side-by-side comparison ────────────────
hotels = sorted(aspect_df['hotel'].unique().tolist())

cat_dd_left = widgets.Dropdown(
    options=hotels,
    value=hotels[0],
    description='Hotel A:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
cat_dd_right = widgets.Dropdown(
    options=hotels,
    value=hotels[1] if len(hotels) > 1 else hotels[0],
    description='Hotel B:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

cat_date_range = widgets.SelectionRangeSlider(
    options=pd.date_range(min_date, max_date, freq='MS').tolist(),
    index=(0, len(pd.date_range(min_date, max_date, freq='MS')) - 1),
    description='Date window:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='700px')
)

cat_output  = widgets.Output()
color_map   = {'positive': '#2ca02c', 'neutral': '#aec7e8', 'negative': '#d62728'}

def get_cat_summary(hotel, start, end):
    dff = aspect_df[
        (aspect_df['hotel'] == hotel) &
        (aspect_df['date'] >= start) &
        (aspect_df['date'] <= end)
    ].copy()
    dff['category'] = dff['aspect'].apply(map_category)
    categorized = dff.dropna(subset=['category'])

    summary = (
        categorized.groupby(['category', 'sentiment'])
        .size().reset_index(name='count')
    )
    total = summary.groupby('category')['count'].transform('sum')
    summary['pct']   = (summary['count'] / total * 100).round(1)
    summary['total'] = total
    return summary, categorized

def plot_cat_comparison(_):
    cat_output.clear_output(wait=True)
    with cat_output:
        h_a   = cat_dd_left.value
        h_b   = cat_dd_right.value
        start = pd.Timestamp(cat_date_range.value[0])
        end   = pd.Timestamp(cat_date_range.value[1])

        # ── Bar charts ────────────────────────────────────────
        fig_bar = make_subplots(
            rows=1, cols=2,
            subplot_titles=[f'<b>{h_a}</b>', f'<b>{h_b}</b>'],
        )

        for col, hotel in enumerate([h_a, h_b], start=1):
            summary, _ = get_cat_summary(hotel, start, end)
            sort_order = (
                summary[summary['sentiment'] == 'positive']
                .sort_values('pct', ascending=True)['category'].tolist()
            )
            for sentiment, color in color_map.items():
                s = summary[summary['sentiment'] == sentiment]
                fig_bar.add_trace(go.Bar(
                    x=s['pct'], y=s['category'],
                    orientation='h',
                    name=sentiment,
                    marker_color=color,
                    text=s['pct'].astype(str) + '%',
                    textposition='inside',
                    showlegend=(col == 1),
                    hovertemplate=f'{hotel}<br>%{{y}}<br>%{{x:.1f}}%<extra></extra>'
                ), row=1, col=col)

            # n= annotations
            for category in summary['category'].unique():
                n = summary[summary['category'] == category]['total'].iloc[0]
                fig_bar.add_annotation(
                    x=103, y=category,
                    text=f'n={n}',
                    showarrow=False,
                    font=dict(size=10, color='gray'),
                    xanchor='left',
                    row=1, col=col
                )

        fig_bar.update_layout(
            barmode='stack',
            title='<b>Category Sentiment Comparison</b>',
            height=450,
            plot_bgcolor='white',
            legend_title='Sentiment',
        )
        fig_bar.update_xaxes(range=[0, 100], title_text='% of mentions')
        fig_bar.show()

        # ── Heatmaps ──────────────────────────────────────────
        fig_heat = make_subplots(
            rows=1, cols=2,
            subplot_titles=[f'<b>{h_a}</b>', f'<b>{h_b}</b>'],
        )

        for col, hotel in enumerate([h_a, h_b], start=1):
            _, categorized = get_cat_summary(hotel, start, end)
            hd = (
                categorized.groupby(['category', 'stars'])['compound']
                .mean().reset_index()
                .pivot(index='category', columns='stars', values='compound')
                .fillna(0)
            )
            fig_heat.add_trace(go.Heatmap(
                z=hd.values,
                x=[f'{s}⭐' for s in hd.columns],
                y=hd.index.tolist(),
                colorscale=[
                    [0.0, '#d62728'],
                    [0.5, '#f7f7f7'],
                    [1.0, '#2ca02c']
                ],
                zmid=0,
                text=hd.values.round(2),
                texttemplate='%{text}',
                showscale=(col == 1),
                hovertemplate=f'{hotel}<br>%{{y}}<br>Stars: %{{x}}<br>Score: %{{z:.2f}}<extra></extra>',
            ), row=1, col=col)

        fig_heat.update_layout(
            title='<b>Category Sentiment by Star Rating — Comparison</b>',
            height=450,
            plot_bgcolor='white',
        )
        fig_heat.show()

# ── Wire up & render ──────────────────────────────────────────
for w in [cat_dd_left, cat_dd_right, cat_date_range]:
    w.observe(plot_cat_comparison, names='value')

display(
    widgets.HTML('<b>Compare category sentiment across hotels:</b>'),
    widgets.HBox([cat_dd_left, cat_dd_right]),
    cat_date_range,
    cat_output
)
plot_cat_comparison(None)

HTML(value='<b>Compare category sentiment across hotels:</b>')

SelectionRangeSlider(description='Date window:', index=(0, 177), layout=Layout(width='700px'), options=(Timest…

Output()

In [36]:
# ── Category positivity over time ─────────────────────────────
cat_time_dd_left = widgets.Dropdown(
    options=hotels,
    value=hotels[0],
    description='Hotel A:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
cat_time_dd_right = widgets.Dropdown(
    options=hotels,
    value=hotels[1] if len(hotels) > 1 else hotels[0],
    description='Hotel B:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
cat_time_gran = widgets.Dropdown(
    options=[('Monthly', 'ME'), ('Quarterly', 'QE'), ('Yearly', 'YE')],
    value='QE',
    description='Granularity:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)
cat_time_date_range = widgets.SelectionRangeSlider(
    options=pd.date_range(min_date, max_date, freq='MS').tolist(),
    index=(0, len(pd.date_range(min_date, max_date, freq='MS')) - 1),
    description='Date window:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='700px')
)
cat_time_output = widgets.Output()

def plot_cat_time(_):
    cat_time_output.clear_output(wait=True)
    with cat_time_output:
        h_a   = cat_time_dd_left.value
        h_b   = cat_time_dd_right.value
        gran  = cat_time_gran.value
        start = pd.Timestamp(cat_time_date_range.value[0])
        end   = pd.Timestamp(cat_time_date_range.value[1])
        gran_label = {'ME': 'Monthly', 'QE': 'Quarterly', 'YE': 'Yearly'}[gran]

        # ── Build categorized df for both hotels ──────────────
        both = aspect_df[
            (aspect_df['hotel'].isin([h_a, h_b])) &
            (aspect_df['date'] >= start) &
            (aspect_df['date'] <= end)
        ].copy()
        both['category'] = both['aspect'].apply(map_category)
        both = both.dropna(subset=['category'])

        categories = sorted(both['category'].unique().tolist())
        colors     = px.colors.qualitative.D3
        color_map  = {cat: colors[i % len(colors)]
                      for i, cat in enumerate(categories)}

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[f'<b>{h_a}</b>', f'<b>{h_b}</b>'],
            shared_yaxes=True
        )

        def smooth(series, window=3):
            return [
                series.iloc[max(0, i-window+1):i+1].mean()
                for i in range(len(series))
            ]

        legend_added = set()

        for col, hotel in enumerate([h_a, h_b], start=1):
            dff = both[both['hotel'] == hotel].copy().set_index('date')

            for category in categories:
                cat_df = dff[dff['category'] == category]
                color  = color_map[category]
                show_legend = category not in legend_added

                if cat_df.empty:
                    if show_legend:
                        fig.add_trace(go.Scatter(
                            x=[], y=[],
                            name=category,
                            mode='lines',
                            line=dict(color=color, width=2.5),
                            legendgroup=category,
                            showlegend=True,
                        ), row=1, col=col)
                        legend_added.add(category)
                    continue

                agg = (
                    cat_df.resample(gran)['compound']
                    .agg(positivity=lambda x: (x > 0.05).mean() * 100)
                    .reset_index()
                )
                agg.columns = ['date', 'positivity']
                agg = agg[agg['positivity'].notna()]

                if len(agg) < 2:
                    continue

                agg['smoothed'] = smooth(agg['positivity'])

                # raw dots
                fig.add_trace(go.Scatter(
                    x=agg['date'], y=agg['positivity'],
                    mode='markers',
                    marker=dict(color=color, size=5, opacity=0.3),
                    legendgroup=category,
                    showlegend=False,
                    hovertemplate=(f'{category}<br>%{{x|%b %Y}}<br>'
                                   f'Raw: %{{y:.0f}}%<extra></extra>')
                ), row=1, col=col)

                # smoothed line
                fig.add_trace(go.Scatter(
                    x=agg['date'], y=agg['smoothed'],
                    mode='lines',
                    name=category,
                    line=dict(color=color, width=2.5),
                    legendgroup=category,
                    showlegend=show_legend,
                    hovertemplate=(f'{category}<br>%{{x|%b %Y}}<br>'
                                   f'Smoothed: %{{y:.0f}}%<extra></extra>')
                ), row=1, col=col)

                legend_added.add(category)

        fig.update_yaxes(
            title_text='% Positive Mentions',
            range=[0, 105],
            showgrid=True,
            gridcolor='rgba(200,200,200,0.4)',
            ticksuffix='%',
            showline=True, linecolor='black',
            row=1, col=1
        )
        fig.update_xaxes(
            showgrid=True,
            gridcolor='rgba(200,200,200,0.2)',
            showline=True, linecolor='black',
        )
        fig.add_hline(
            y=50, line_dash='dash', line_color='gray',
            line_width=1, opacity=0.5,
            annotation_text='50%', annotation_position='left'
        )
        fig.update_layout(
            title=dict(
                text=(f'<b>Category Positivity Over Time</b> — {gran_label}<br>'
                      f'<sup>3-period smoothed | dots = raw | '
                      f'click legend to toggle</sup>'),
                font=dict(size=14)
            ),
            plot_bgcolor='white',
            paper_bgcolor='white',
            height=480,
            hovermode='x unified',
            legend=dict(
                title='Category',
                orientation='v',
                yanchor='top', y=1,
                xanchor='left', x=1.02
            ),
        )
        fig.show()

for w in [cat_time_dd_left, cat_time_dd_right,
          cat_time_gran, cat_time_date_range]:
    w.observe(plot_cat_time, names='value')

display(
    widgets.HTML('<b>Category Positivity Over Time:</b>'),
    widgets.HBox([cat_time_dd_left, cat_time_dd_right, cat_time_gran]),
    cat_time_date_range,
    cat_time_output
)
plot_cat_time(None)

HTML(value='<b>Category Positivity Over Time:</b>')

SelectionRangeSlider(description='Date window:', index=(0, 177), layout=Layout(width='700px'), options=(Timest…

Output()

In [37]:


# ── Change this to switch hotel ───────────────────────────────
SELECTED_HOTEL = 'Banff Caribou Lodge & Spa'

# ════════════════════════════════════════════════════════════
# FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════
def build_hotel_features(hotel):
    # ── Per-review category sentiment pivot ───────────────────
    asp = aspect_df[aspect_df['hotel'] == hotel].copy()
    asp['category'] = asp['aspect'].apply(map_category)

    cat_pivot = (
        asp.dropna(subset=['category'])
        .groupby(['reviewer', 'date', 'category'])['compound']
        .mean()
        .unstack(fill_value=0)
        .reset_index()
    )
    cat_pivot.columns.name = None
    cat_pivot.columns = ['reviewer', 'date'] + [
        f'cat_{c.lower().replace(" & ", "_").replace(" ", "_")}'
        for c in cat_pivot.columns[2:]
    ]

    # ── Base review-level features ────────────────────────────
    rev = df[df['hotel'] == hotel].copy()
    rev['text_length']        = rev['text'].fillna('').apply(len)
    rev['is_local_guide']     = rev['is_local_guide'].map(
                                 {True: 1, False: 0, 'true': 1, 'false': 0, 1: 1, 0: 0}
                                ).fillna(0).astype(int)
    rev['reviewer_reviews']   = pd.to_numeric(rev['reviewer_reviews'], errors='coerce').fillna(0)
    rev['reviewer_photos']    = pd.to_numeric(rev['reviewer_photos'],  errors='coerce').fillna(0)
    rev['has_owner_response'] = rev['owner_response'].notna().astype(int)
    rev['is_winter']          = rev['date'].dt.month.isin([12, 1, 2, 3]).astype(int)
    rev['is_google']          = (rev['source'] == 'google').astype(int)

    # ── Fill NaN categoricals with 'Unknown' before dummying ──
    rev['trip_type']    = rev['trip_type'].fillna('Unknown')
    rev['travel_group'] = rev['travel_group'].fillna('Unknown')

    # ── Check fill rates ──────────────────────────────────────
    print(f"\n  trip_type coverage:    {(rev['trip_type'] != 'Unknown').mean()*100:.0f}%")
    print(f"  travel_group coverage: {(rev['travel_group'] != 'Unknown').mean()*100:.0f}%")
    print(f"\n  trip_type values:\n{rev['trip_type'].value_counts().to_string()}")
    print(f"\n  travel_group values:\n{rev['travel_group'].value_counts().to_string()}")

    # dummies — Unknown is the implicit reference category (dropped)
    trip_dummies  = pd.get_dummies(
        rev['trip_type'],    prefix='trip',  drop_first=False
    ).astype(int)
    trip_dummies  = trip_dummies.drop(
        columns=[c for c in trip_dummies.columns if 'Unknown' in c], errors='ignore'
    )

    group_dummies = pd.get_dummies(
        rev['travel_group'], prefix='group', drop_first=False
    ).astype(int)
    group_dummies = group_dummies.drop(
        columns=[c for c in group_dummies.columns if 'Unknown' in c], errors='ignore'
    )

    base = pd.concat([
        rev[['reviewer', 'date', 'stars', 'text_length', 'is_local_guide',
             'reviewer_reviews', 'reviewer_photos', 'has_owner_response',
             'is_winter', 'is_google']],
        trip_dummies,
        group_dummies,
    ], axis=1)

    # ── Merge category sentiment ──────────────────────────────
    feat = base.merge(cat_pivot, on=['reviewer', 'date'], how='left').fillna(0)

    # ── Target ────────────────────────────────────────────────
    feat['positive'] = (feat['stars'] >= 4).astype(int)

    feature_cols = [c for c in feat.columns if c not in [
        'reviewer', 'date', 'stars', 'positive'
    ]]

    X = feat[feature_cols].astype(float)
    y = feat['positive']

    print(f"\n✅ {hotel}")
    print(f"   {len(feat):,} reviews | {len(feature_cols)} features")
    print(f"   Positive (4-5★): {y.mean()*100:.0f}%  "
          f"Negative (1-3★): {(1-y).mean()*100:.0f}%")
    print(f"\n   Features: {feature_cols}")

    return X, y, feature_cols



X, y, feature_cols = build_hotel_features(SELECTED_HOTEL)

X, y, feature_cols = build_hotel_features(SELECTED_HOTEL)

# ════════════════════════════════════════════════════════════
# LOGISTIC REGRESSION — STATSMODELS
# ════════════════════════════════════════════════════════════

# drop any zero-variance columns
X = X.loc[:, X.std() > 0]
feature_cols = X.columns.tolist()

X_sm = sm.add_constant(X)

model = sm.Logit(y, X_sm)
result = model.fit(method='bfgs', maxiter=500, disp=False)

print(result.summary(
    xname=['const'] + feature_cols,
    title=f'Logistic Regression — {SELECTED_HOTEL} — P(positive review)'
))

# ════════════════════════════════════════════════════════════
# TIDY COEFFICIENT TABLE
# ════════════════════════════════════════════════════════════
def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '.'
    return ''

coef_table = pd.DataFrame({
    'feature':     feature_cols,
    'coef':        result.params[1:].values,
    'odds_ratio':  np.exp(result.params[1:].values),
    'std_err':     result.bse[1:].values,
    'z_stat':      result.tvalues[1:].values,
    'p_value':     result.pvalues[1:].values,
    'ci_lower':    result.conf_int()[1:][0].values,
    'ci_upper':    result.conf_int()[1:][1].values,
}).sort_values('p_value')

coef_table['sig']      = coef_table['p_value'].apply(sig_stars)
coef_table['ci_lower_or'] = np.exp(coef_table['ci_lower'])
coef_table['ci_upper_or'] = np.exp(coef_table['ci_upper'])

with pd.option_context('display.max_rows', None, 'display.float_format', '{:.4f}'.format):
    print(f"\nCoefficients sorted by p-value:")
    print(coef_table[[
        'feature', 'coef', 'odds_ratio', 'std_err', 'z_stat', 'p_value', 'sig'
    ]].to_string(index=False))
    print("\nSignificance: *** p<0.001  ** p<0.01  * p<0.05  . p<0.10")

# ════════════════════════════════════════════════════════════
# ODDS RATIO PLOT — significant features only
# ════════════════════════════════════════════════════════════
sig_coefs = coef_table.sort_values('odds_ratio', ascending=True)

if len(sig_coefs) == 0:
    print("\n⚠️ No significant features at p<0.10 — try more data or fewer features")
else:
    colors  = ['#d62728' if o < 1 else '#2ca02c' for o in sig_coefs['odds_ratio']]
    opacity = [1.0 if p < 0.05 else 0.6 for p in sig_coefs['p_value']]

    fig = go.Figure()

    # CI lines
    for _, r in sig_coefs.iterrows():
        fig.add_shape(type='line',
            x0=r['ci_lower_or'], x1=r['ci_upper_or'],
            y0=r['feature'],     y1=r['feature'],
            line=dict(color='gray', width=1.5)
        )

    # OR dots
    fig.add_trace(go.Scatter(
        x=sig_coefs['odds_ratio'],
        y=sig_coefs['feature'],
        mode='markers',
        marker=dict(color=colors, size=10, opacity=opacity,
                    line=dict(color='white', width=1)),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Odds Ratio: %{x:.3f}<br>'
            'p: %{customdata[0]:.4f} %{customdata[1]}<extra></extra>'
        ),
        customdata=list(zip(sig_coefs['p_value'], sig_coefs['sig']))
    ))

    fig.add_vline(x=1, line_width=1.5, line_dash='dash', line_color='black')

    fig.update_layout(
        title=dict(
            text=(f'<b>Odds Ratios — Significant Features</b> — {SELECTED_HOTEL}<br>'
                  f'<sup>OR > 1 = increases P(positive review) | '
                  f'faded = p<0.10, solid = p<0.05 | bars = 95% CI</sup>'),
            font=dict(size=14)
        ),
        xaxis_title='Odds Ratio',
        yaxis_title='',
        height=max(400, len(sig_coefs) * 30),
        plot_bgcolor='white',
        xaxis=dict(showgrid=True, gridcolor='rgba(200,200,200,0.4)'),
    )
    fig.show()

    print(f"\nModel fit:  Pseudo R² = {result.prsquared:.3f}  |  "
          f"AIC = {result.aic:.1f}  |  "
          f"Log-likelihood = {result.llf:.1f}")


  trip_type coverage:    19%
  travel_group coverage: 18%

  trip_type values:
trip_type
Unknown     1526
Vacation     341
Business      20

  travel_group values:
travel_group
Unknown    1540
Couple      154
Family      117
Friends      56
Solo         20

✅ Banff Caribou Lodge & Spa
   1,887 reviews | 19 features
   Positive (4-5★): 76%  Negative (1-3★): 24%

   Features: ['text_length', 'is_local_guide', 'reviewer_reviews', 'reviewer_photos', 'has_owner_response', 'is_winter', 'is_google', 'trip_Business', 'trip_Vacation', 'group_Couple', 'group_Family', 'group_Friends', 'group_Solo', 'cat_amenities', 'cat_food_breakfast', 'cat_location', 'cat_rooms', 'cat_service', 'cat_value']

  trip_type coverage:    19%
  travel_group coverage: 18%

  trip_type values:
trip_type
Unknown     1526
Vacation     341
Business      20

  travel_group values:
travel_group
Unknown    1540
Couple      154
Family      117
Friends      56
Solo         20

✅ Banff Caribou Lodge & Spa
   1,887 reviews | 19


Model fit:  Pseudo R² = 0.234  |  AIC = 1641.1  |  Log-likelihood = -801.5


In [38]:


hotels      = sorted(df['hotel'].unique().tolist())
all_results = {}  # store for later use

for hotel in hotels:
    print("\n" + "="*65)
    print(f"  {hotel}")
    print("="*65)

    try:
        X, y, feature_cols = build_hotel_features(hotel)

        if len(X) < 30:
            print(f"⚠️  Skipping — only {len(X)} samples")
            continue
        if y.nunique() < 2:
            print(f"⚠️  Skipping — no variance in target")
            continue

        # ── Standardise continuous features ──────────────────
        binary_cols     = [c for c in feature_cols if X[c].nunique() == 2]
        continuous_cols = [c for c in feature_cols if c not in binary_cols]

        X_scaled = X.copy()
        if continuous_cols:
            scaler = StandardScaler()
            X_scaled[continuous_cols] = scaler.fit_transform(X[continuous_cols])

        # drop zero variance after scaling
        X_scaled = X_scaled.loc[:, X_scaled.std() > 0]
        feature_cols_clean = X_scaled.columns.tolist()

        # ── Fit ───────────────────────────────────────────────
        X_sm   = sm.add_constant(X_scaled)
        model  = sm.Logit(y, X_sm)
        result = model.fit(method='bfgs', maxiter=500, disp=False)

        # ── Summary ───────────────────────────────────────────
        print(result.summary(
            xname=['const'] + feature_cols_clean,
            title=f'Logistic Regression — {hotel}'
        ))

        # ── Tidy table ────────────────────────────────────────
        coef_table = pd.DataFrame({
            'feature':    feature_cols_clean,
            'coef':       result.params[1:].values,
            'odds_ratio': np.exp(result.params[1:].values),
            'std_err':    result.bse[1:].values,
            'z_stat':     result.tvalues[1:].values,
            'p_value':    result.pvalues[1:].values,
            'ci_lower_or': np.exp(result.conf_int()[1:][0].values),
            'ci_upper_or': np.exp(result.conf_int()[1:][1].values),
        }).sort_values('p_value')

        coef_table['sig'] = coef_table['p_value'].apply(sig_stars)

        with pd.option_context('display.max_rows', None,
                               'display.float_format', '{:.4f}'.format):
            print(f"\nCoefficients sorted by p-value:")
            print(coef_table[[
                'feature','coef','odds_ratio','std_err','z_stat','p_value','sig'
            ]].to_string(index=False))
            print("\nSignificance: *** p<0.001  ** p<0.01  * p<0.05  . p<0.10")
            print(f"\nPseudo R²: {result.prsquared:.3f}  |  "
                  f"AIC: {result.aic:.1f}  |  "
                  f"N: {len(X)}")

        # ── Odds ratio plot — significant only ────────────────
        sig_coefs = coef_table.sort_values(
            'odds_ratio', ascending=True
        )

        if len(sig_coefs) == 0:
            print(f"\n⚠️  No significant features at p<0.10")
        else:
            colors  = ['#d62728' if o < 1 else '#2ca02c'
                       for o in sig_coefs['odds_ratio']]
            opacity = [1.0 if p < 0.05 else 0.6
                       for p in sig_coefs['p_value']]

            fig = go.Figure()

            for _, r in sig_coefs.iterrows():
                fig.add_shape(type='line',
                    x0=r['ci_lower_or'], x1=r['ci_upper_or'],
                    y0=r['feature'],     y1=r['feature'],
                    line=dict(color='gray', width=1.5)
                )

            fig.add_trace(go.Scatter(
                x=sig_coefs['odds_ratio'],
                y=sig_coefs['feature'],
                mode='markers',
                marker=dict(
                    color=colors, size=10, opacity=opacity,
                    line=dict(color='white', width=1)
                ),
                hovertemplate=(
                    '<b>%{y}</b><br>'
                    'OR: %{x:.3f}<br>'
                    'p: %{customdata[0]:.4f} %{customdata[1]}<extra></extra>'
                ),
                customdata=list(zip(sig_coefs['p_value'], sig_coefs['sig']))
            ))

            fig.add_vline(x=1, line_width=1.5, line_dash='dash',
                          line_color='black')

            fig.update_layout(
                title=dict(
                    text=(f'<b>Odds Ratios</b> — {hotel}<br>'
                          f'<sup>OR>1 increases P(positive) | '
                          f'faded=p<0.10, solid=p<0.05 | bars=95% CI</sup>'),
                    font=dict(size=13)
                ),
                xaxis_title='Odds Ratio',
                yaxis_title='',
                height=max(350, len(sig_coefs) * 32),
                plot_bgcolor='white',
                xaxis=dict(showgrid=True, gridcolor='rgba(200,200,200,0.4)'),
            )
            fig.show()

        # ── Store results for cross-hotel comparison later ────
        all_results[hotel] = {
            'result':     result,
            'coef_table': coef_table,
            'n':          len(X),
            'pseudo_r2':  result.prsquared,
            'aic':        result.aic,
        }

    except Exception as e:
        print(f"❌ Failed: {e}")
        continue

# ── Cross-hotel model fit summary ─────────────────────────────
print("\n" + "="*65)
print("  MODEL FIT SUMMARY — ALL HOTELS")
print("="*65)
fit_summary = pd.DataFrame({
    h: {
        'N':          v['n'],
        'Pseudo R²':  round(v['pseudo_r2'], 3),
        'AIC':        round(v['aic'], 1),
    }
    for h, v in all_results.items()
}).T

with pd.option_context('display.max_rows', None):
    print(fit_summary.to_string())


  Banff Caribou Lodge & Spa

  trip_type coverage:    19%
  travel_group coverage: 18%

  trip_type values:
trip_type
Unknown     1526
Vacation     341
Business      20

  travel_group values:
travel_group
Unknown    1540
Couple      154
Family      117
Friends      56
Solo         20

✅ Banff Caribou Lodge & Spa
   1,887 reviews | 19 features
   Positive (4-5★): 76%  Negative (1-3★): 24%

   Features: ['text_length', 'is_local_guide', 'reviewer_reviews', 'reviewer_photos', 'has_owner_response', 'is_winter', 'is_google', 'trip_Business', 'trip_Vacation', 'group_Couple', 'group_Family', 'group_Friends', 'group_Solo', 'cat_amenities', 'cat_food_breakfast', 'cat_location', 'cat_rooms', 'cat_service', 'cat_value']
               Logistic Regression — Banff Caribou Lodge & Spa                
Dep. Variable:               positive   No. Observations:                 1887
Model:                          Logit   Df Residuals:                     1868
Method:                           MLE   Df


  Banff Inn

  trip_type coverage:    19%
  travel_group coverage: 19%

  trip_type values:
trip_type
Unknown     720
Vacation    163
Business      8

  travel_group values:
travel_group
Unknown    725
Family      73
Couple      53
Friends     30
Solo        10

✅ Banff Inn
   891 reviews | 19 features
   Positive (4-5★): 80%  Negative (1-3★): 20%

   Features: ['text_length', 'is_local_guide', 'reviewer_reviews', 'reviewer_photos', 'has_owner_response', 'is_winter', 'is_google', 'trip_Business', 'trip_Vacation', 'group_Couple', 'group_Family', 'group_Friends', 'group_Solo', 'cat_amenities', 'cat_food_breakfast', 'cat_location', 'cat_rooms', 'cat_service', 'cat_value']
                       Logistic Regression — Banff Inn                        
Dep. Variable:               positive   No. Observations:                  891
Model:                          Logit   Df Residuals:                      872
Method:                           MLE   Df Model:                           18
Date:


  Banff Ptarmigan Inn

  trip_type coverage:    19%
  travel_group coverage: 19%

  trip_type values:
trip_type
Unknown     1073
Vacation     248
Business      10

  travel_group values:
travel_group
Unknown    1072
Couple      113
Family       89
Friends      43
Solo         14

✅ Banff Ptarmigan Inn
   1,331 reviews | 19 features
   Positive (4-5★): 77%  Negative (1-3★): 23%

   Features: ['text_length', 'is_local_guide', 'reviewer_reviews', 'reviewer_photos', 'has_owner_response', 'is_winter', 'is_google', 'trip_Business', 'trip_Vacation', 'group_Couple', 'group_Family', 'group_Friends', 'group_Solo', 'cat_amenities', 'cat_food_breakfast', 'cat_location', 'cat_rooms', 'cat_service', 'cat_value']
                  Logistic Regression — Banff Ptarmigan Inn                   
Dep. Variable:               positive   No. Observations:                 1331
Model:                          Logit   Df Residuals:                     1313
Method:                           MLE   Df Model:     


  Mount Royal Hotel

  trip_type coverage:    22%
  travel_group coverage: 21%

  trip_type values:
trip_type
Unknown     929
Vacation    235
Business     22

  travel_group values:
travel_group
Unknown    942
Couple     121
Family      73
Friends     27
Solo        23

✅ Mount Royal Hotel
   1,186 reviews | 19 features
   Positive (4-5★): 83%  Negative (1-3★): 17%

   Features: ['text_length', 'is_local_guide', 'reviewer_reviews', 'reviewer_photos', 'has_owner_response', 'is_winter', 'is_google', 'trip_Business', 'trip_Vacation', 'group_Couple', 'group_Family', 'group_Friends', 'group_Solo', 'cat_amenities', 'cat_food_breakfast', 'cat_location', 'cat_rooms', 'cat_service', 'cat_value']
                   Logistic Regression — Mount Royal Hotel                    
Dep. Variable:               positive   No. Observations:                 1186
Model:                          Logit   Df Residuals:                     1167
Method:                           MLE   Df Model:                 


  The Rundlestone Lodge

  trip_type coverage:    17%
  travel_group coverage: 17%

  trip_type values:
trip_type
Unknown     840
Vacation    165
Business     11

  travel_group values:
travel_group
Unknown    844
Couple      71
Family      68
Friends     24
Solo         9

✅ The Rundlestone Lodge
   1,016 reviews | 19 features
   Positive (4-5★): 80%  Negative (1-3★): 20%

   Features: ['text_length', 'is_local_guide', 'reviewer_reviews', 'reviewer_photos', 'has_owner_response', 'is_winter', 'is_google', 'trip_Business', 'trip_Vacation', 'group_Couple', 'group_Family', 'group_Friends', 'group_Solo', 'cat_amenities', 'cat_food_breakfast', 'cat_location', 'cat_rooms', 'cat_service', 'cat_value']
                 Logistic Regression — The Rundlestone Lodge                  
Dep. Variable:               positive   No. Observations:                 1016
Model:                          Logit   Df Residuals:                      997
Method:                           MLE   Df Model:         


  MODEL FIT SUMMARY — ALL HOTELS
                                N  Pseudo R²     AIC
Banff Caribou Lodge & Spa  1887.0      0.234  1641.1
Banff Inn                   891.0      0.245   710.5
Banff Ptarmigan Inn        1331.0      0.180  1208.7
Mount Royal Hotel          1186.0      0.186   919.5
The Rundlestone Lodge      1016.0      0.280   775.7


In [39]:


# ── Extended stopwords ────────────────────────────────────────
CLOUD_STOPWORDS = set(STOPWORDS) | {
    'hotel', 'stay', 'stayed', 'room', 'rooms', 'place', 'banff',
    'canmore', 'will', 'would', 'could', 'get', 'got', 'one', 'also',
    'really', 'even', 'back', 'us', 'went', 'said', 'told', 'came',
    'time', 'day', 'night', 'trip', 'visit', 'well', 'still', 'made',
    'make', 'just', 'like', 'good', 'great', 'nice', 'wonderful',
    'amazing', 'excellent', 'perfect', 'beautiful', 'lovely', 'awesome',
}

# ── Widgets ───────────────────────────────────────────────────
wc_hotel_dd = widgets.Dropdown(
    options=sorted(df['hotel'].unique().tolist()),
    description='Hotel:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

wc_sentiment_dd = widgets.Dropdown(
    options=[('All reviews', 'all'), ('Positive only (4-5★)', 'positive'),
             ('Negative only (1-3★)', 'negative')],
    value='all',
    description='Filter:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)

wc_output = widgets.Output()

def plot_wordcloud(_):
    wc_output.clear_output(wait=True)
    with wc_output:
        hotel     = wc_hotel_dd.value
        sentiment = wc_sentiment_dd.value

        dff = df[df['hotel'] == hotel].dropna(subset=['text']).copy()

        if sentiment == 'positive':
            dff = dff[dff['stars'] >= 4]
        elif sentiment == 'negative':
            dff = dff[dff['stars'] <= 3]

        if dff.empty:
            print("No reviews match selection.")
            return

        text = ' '.join(dff['text'].astype(str).tolist())
        n    = len(dff)

        # ── Generate two clouds: positive & negative context ──
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.patch.set_facecolor('white')

        sentiment_label = {
            'all': 'All Reviews',
            'positive': 'Positive Reviews (4-5★)',
            'negative': 'Negative Reviews (1-3★)'
        }[sentiment]

        fig.suptitle(
            f'{hotel}  —  Word Cloud  —  {sentiment_label}  (n={n:,})',
            fontsize=14, fontweight='bold', y=1.01
        )

        # ── Left: frequency cloud ─────────────────────────────
        wc_freq = WordCloud(
            width=700, height=400,
            background_color='white',
            stopwords=CLOUD_STOPWORDS,
            colormap='Blues',
            max_words=100,
            collocations=True,        # keeps bigrams together
            min_word_length=3,
        ).generate(text)

        axes[0].imshow(wc_freq, interpolation='bilinear')
        axes[0].axis('off')
        axes[0].set_title('Most Frequent Terms', fontsize=12, pad=10)

        # ── Right: sentiment-coloured cloud ───────────────────
        # colour words by their VADER sentiment
        def sentiment_color(word, font_size, position, orientation,
                            random_state=None, **kwargs):
            score = analyzer.polarity_scores(word)['compound']
            if score >= 0.05:
                # green shades
                intensity = int(100 + score * 100)
                return f'rgb(0, {intensity}, 0)'
            elif score <= -0.05:
                # red shades
                intensity = int(100 + abs(score) * 100)
                return f'rgb({intensity}, 0, 0)'
            else:
                return 'rgb(128, 128, 128)'

        wc_sent = WordCloud(
            width=700, height=400,
            background_color='white',
            stopwords=CLOUD_STOPWORDS,
            max_words=100,
            collocations=True,
            min_word_length=3,
        ).generate(text)

        axes[1].imshow(
            wc_sent.recolor(color_func=sentiment_color),
            interpolation='bilinear'
        )
        axes[1].axis('off')
        axes[1].set_title(
            'Sentiment-Coloured  (green=positive, red=negative, grey=neutral)',
            fontsize=12, pad=10
        )

        plt.tight_layout()
        plt.show()

# ── Wire up & render ──────────────────────────────────────────
for w in [wc_hotel_dd, wc_sentiment_dd]:
    w.observe(plot_wordcloud, names='value')

display(
    widgets.HTML('<b>Word Cloud — Customer Reviews:</b>'),
    widgets.HBox([wc_hotel_dd, wc_sentiment_dd]),
    wc_output
)
plot_wordcloud(None)

HTML(value='<b>Word Cloud — Customer Reviews:</b>')

Output()